# Clase guiada: limpieza y estructura de datos de RRHH
**Dataset:** IBM HR Employee Attrition (1.470 empleados, 35 columnas) — version "ensuciada" a proposito.
**Duracion:** 40 minutos · **Herramientas:** Python + pandas

**Pregunta de negocio que guia todo:** *¿que caracteristicas tienen las personas que se van de la empresa?*
Hoy NO respondemos esa pregunta. Hoy dejamos los datos en condiciones de poder responderla.

| Bloque | Minutos | Que hacemos |
|---|---|---|
| 0. Contexto | 0–5 | Por que limpiar y como vamos a trabajar |
| 1. Cargar y mirar | 5–10 | Primer contacto con los datos |
| 2. Diagnosticar | 10–17 | Encontrar los problemas **antes** de tocar nada |
| 3. Limpiar | 17–30 | Duplicados, texto, tipos, valores imposibles, nulos |
| 4. Estructurar | 30–35 | Escalas legibles y variables nuevas |
| 5. Validar y guardar | 35–40 | Chequeos automaticos + bitacora de decisiones |

> Las celdas marcadas con ✏️ son **para vos**. Las demas las vemos juntos.

## 0. Tres reglas de trabajo
1. **Nunca pisamos el original.** Trabajamos sobre una copia; el archivo crudo queda intacto.
2. **Primero diagnostico, despues limpieza.** Si arreglas lo primero que ves, te perdes lo demas.
3. **Toda decision queda escrita.** "Imputé el sueldo con la mediana del nivel" es una decision de negocio, no solo de codigo. Alguien te la va a preguntar.

## 1. Cargar y mirar (min 5–10)

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# Si todavia no generaste el archivo sucio:
#   python ensuciar_datos.py WA_Fn-UseC_-HR-Employee-Attrition.csv
crudo = pd.read_csv("HR_sucio.csv", encoding="utf-8-sig")  # utf-8-sig: el CSV original trae un caracter invisible (BOM) al inicio
crudo.shape

In [ ]:
crudo.head()

In [ ]:
crudo.info()

**Pregunta para charlar:** mirando `info()`, ¿hay alguna columna cuyo *tipo* no tenga sentido?
<details><summary>Pista</summary>¿Un sueldo deberia ser texto?</details>

## 2. Diagnosticar (min 10–17)
Objetivo: una lista de problemas. **Todavia no arreglamos nada.**

In [ ]:
# Nulos por columna (solo las que tienen alguno)
nulos = crudo.isna().sum()
nulos[nulos > 0]

In [ ]:
# ✏️ Ejercicio 1
# a) ¿Cuantas filas estan duplicadas exactamente? (pista: .duplicated())
# b) ¿Que columnas tienen UN solo valor en todo el dataset? (pista: .nunique())
#    ¿Sirven para algo esas columnas?


In [ ]:
# ✏️ Ejercicio 2
# Recorre las columnas de texto y mira sus valores distintos con .value_counts().
# Anota: ¿que inconsistencias encontras?
columnas_texto = crudo.select_dtypes(exclude="number").columns


In [ ]:
# Rangos de las numericas: ¿algo imposible?
crudo.describe().T[["min", "max"]]

**Lista de problemas que deberiamos tener a esta altura** (completar juntos antes de seguir):
- [ ] Filas duplicadas
- [ ] Columnas constantes (`EmployeeCount`, `Over18`, `StandardHours`)
- [ ] `Department`, `Gender`, `Attrition`, `OverTime` escritos de muchas formas
- [ ] `MonthlyIncome` guardado como texto (`"$5,993"`)
- [ ] Edades imposibles (`Age` = -1, 4, 150, 999)
- [ ] Nulos en `MonthlyIncome`, `DistanceFromHome`, `Department`

## 3. Limpiar (min 17–30)
Trabajamos sobre `df` (copia) y registramos cada decision en `bitacora`.

In [ ]:
df = crudo.copy()
bitacora = []

def registrar(decision):
    bitacora.append(decision)
    print("📝", decision)

### 3.1 Duplicados

In [ ]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
registrar(f"Elimine {antes - len(df)} filas duplicadas exactas")

> **Para pensar:** en un sistema real, ¿alcanza con buscar filas *identicas*? ¿Que pasa si el mismo `EmployeeNumber` aparece dos veces con datos distintos?

In [ ]:
# ✏️ Ejercicio 3
# Elimina las columnas constantes que encontraste en el Ejercicio 1 y registralo en la bitacora.


### 3.2 Texto inconsistente
Receta: 1) sacar espacios, 2) llevar a un formato comun, 3) mapear sinonimos con un diccionario.

In [ ]:
df["Department"] = df["Department"].str.strip().str.lower().replace({
    "sales": "Sales",
    "research & development": "Research & Development",
    "r&d": "Research & Development",
    "human resources": "Human Resources",
    "hr": "Human Resources",
})
registrar("Department: unifiqué mayúsculas, espacios y abreviaturas (R&D, HR)")
df["Department"].value_counts(dropna=False)

In [ ]:
# ✏️ Ejercicio 4
# a) Unifica Gender en dos valores: "Male" y "Female".
# b) Converti Attrition y OverTime a booleano (True/False).
#    Tip: armate un diccionario con TODAS las variantes que viste en el Ejercicio 2.
#    Verifica al final que no quede ningun valor sin mapear.


### 3.3 Numeros guardados como texto

In [ ]:
sueldo_texto = df["MonthlyIncome"].astype("string").str.replace(r"[$,]", "", regex=True)
df["MonthlyIncome"] = pd.to_numeric(sueldo_texto, errors="coerce")
registrar("MonthlyIncome: saque '$' y ',' y lo converti a numero")
df["MonthlyIncome"].describe()

### 3.4 Valores imposibles
**Decision de negocio:** ¿borramos la fila entera o solo el dato malo?
Si borramos la fila perdemos el resto de la informacion de esa persona (que esta bien). Preferimos marcar el dato como faltante.

In [ ]:
# ✏️ Ejercicio 5
# Converti en NaN las edades fuera de un rango laboral razonable (vos elegis el rango y lo justificas).
# ¿Cuantas edades corregiste? Registralo.


### 3.5 Nulos: cada columna merece su propia decision
| Columna | Idea | Por que |
|---|---|---|
| `MonthlyIncome` | Mediana **segun `JobLevel`** | El sueldo depende mucho del nivel; la mediana general seria injusta para un gerente |
| `Department` | Deducirlo de `JobRole` | Casi todos los puestos pertenecen a un solo departamento |
| `Age`, `DistanceFromHome` | Mediana general | Pocos casos, sin una variable que los explique mejor |

Buena practica: **dejar una marca** de que el valor fue imputado.

In [ ]:
df["sueldo_imputado"] = df["MonthlyIncome"].isna()
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(
    df.groupby("JobLevel")["MonthlyIncome"].transform("median")
)
registrar(f"MonthlyIncome: imputé {df['sueldo_imputado'].sum()} nulos con la mediana de su JobLevel")

In [ ]:
# ✏️ Ejercicio 6
# a) Mira pd.crosstab(df["JobRole"], df["Department"]). ¿Que puestos estan en un solo departamento?
# b) Completa los Department nulos usando el JobRole cuando se pueda.
#    ¿Que haces con los "Manager"? (estan en los tres departamentos)
# c) Completa Age y DistanceFromHome con la mediana.


## 4. Estructurar (min 30–35)
Muchas columnas son escalas codificadas con numeros (1 a 4). Para leer y graficar conviene tener la etiqueta,
y para el modelo conviene que pandas sepa que **hay un orden** (Low < Medium < High).

In [ ]:
escalas = {
    "Education": {1: "Below College", 2: "College", 3: "Bachelor", 4: "Master", 5: "Doctor"},
    "EnvironmentSatisfaction": {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "JobSatisfaction":         {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "RelationshipSatisfaction":{1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "JobInvolvement":          {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "WorkLifeBalance":         {1: "Bad", 2: "Good", 3: "Better", 4: "Best"},
    "PerformanceRating":       {1: "Low", 2: "Good", 3: "Excellent", 4: "Outstanding"},
}
for col, etiquetas in escalas.items():
    orden = list(etiquetas.values())
    df[col + "_txt"] = pd.Categorical(df[col].map(etiquetas), categories=orden, ordered=True)

registrar("Agregué versiones con etiqueta (ordenadas) de las escalas 1-4/1-5; mantuve el número original")
df[["JobSatisfaction", "JobSatisfaction_txt"]].head()

In [ ]:
# ✏️ Ejercicio 7 — variables nuevas pensadas desde RRHH
# a) tramo_edad: agrupa Age en rangos (pista: pd.cut). Vos elegis los cortes.
# b) tramo_antiguedad: agrupa YearsAtCompany (ej.: <1, 1-2, 3-5, 6-10, +10).
# c) Inventa UNA variable que, desde tu experiencia en RRHH, creas que explica la rotacion.
#    Ej.: años sin ascenso respecto de la antigüedad. ¡Cuidado con dividir por cero!


> **Para pensar (sin codigo):** `DailyRate`, `HourlyRate` y `MonthlyRate` casi no se correlacionan con `MonthlyIncome`.
> ¿Que significan? Si no podes explicar una columna, anotalo como pregunta abierta antes de usarla en un modelo.
>
> **Etica:** `Gender`, `Age` y `MaritalStatus` son datos sensibles. Sirven para *detectar* inequidades,
> pero hay que pensar dos veces antes de usarlos para *decidir* sobre personas.

## 5. Validar y guardar (min 35–40)
Los chequeos se escriben **una vez** y se corren **cada vez** que cambia el proceso.
Si mañana llega un archivo nuevo, te avisan solos si algo se rompio.

In [ ]:
def chequear(condicion, mensaje):
    print(("OK   " if condicion else "FALLA") + " | " + mensaje)

chequear(not df.duplicated().any(), "sin filas duplicadas")
chequear(df["EmployeeNumber"].is_unique, "un registro por empleado")
chequear(df.drop(columns=[c + "_txt" for c in escalas]).notna().all().all(), "sin nulos (fuera de las etiquetas)")
chequear(df["Age"].between(18, 70).all(), "edades en rango")
chequear(df["Gender"].isin(["Male", "Female"]).all(), "Gender con 2 valores")
chequear(pd.api.types.is_numeric_dtype(df["MonthlyIncome"]), "MonthlyIncome es numerico")

In [ ]:
# ✏️ Ejercicio 8
# Agrega al menos DOS chequeos propios. Ideas:
#  - YearsAtCompany nunca es mayor que TotalWorkingYears
#  - la tasa de rotacion (Attrition) sigue cerca del 16% que tenia el original


In [ ]:
df.to_csv("HR_limpio.csv", index=False)
pd.Series(bitacora, name="decision").to_csv("bitacora_limpieza.csv", index=False)
print(f"Guardado: {df.shape[0]} filas x {df.shape[1]} columnas\n")
print("\n".join(f"{i}. {d}" for i, d in enumerate(bitacora, 1)))

## Cierre
**Tres preguntas para vos:**
1. ¿Cual de las decisiones de hoy te parece mas discutible? ¿Como la defenderias frente a un gerente?
2. Si el dataset tuviera 100 veces mas filas, ¿que cambiarias del proceso?
3. ¿Que columna **falta** para responder "por que se va la gente"? (pensalo desde RRHH, no desde Python)

**Para la proxima (lo haces vos):**
- Subir `HR_limpio.csv` a MySQL (pista: `sqlalchemy.create_engine` + `df.to_sql`) y repetir 2 chequeos en SQL.
- Primer analisis exploratorio: tasa de rotacion por `Department`, `OverTime`, `tramo_edad` y `JobSatisfaction_txt`.
- Traer 3 hallazgos y 1 pregunta que no hayas podido responder.